# Salzberger et al. 2018 Explorer
 
Interactive notebook for exploring paper

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import Layout
from IPython.display import display

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

COLORS = {
    'electron': "#1f26b4", 
    'proton': "#d60303"
}
MeV_MARKERS = {1: 'o', 3: 's', 10: 'd'}  # by energy (MeV)

## Load the data

In [ ]:
df = pd.read_csv('./SalzbergerEtal2018.csv', encoding='utf-8-sig')
df.columns = [c.strip() for c in df.columns]
df

In [ ]:
print("Rows:", len(df))
print("Particle types:", sorted(df['particle type'].unique()))
print("Energies (MeV):", sorted(df['energy (MeV)'].unique()))

## Parameter dictionary

Columns available to plot on the y-axis, with friendly labels and units.

In [ ]:
PARAM_INFO = {
    'I_01':  {
        'label': 'I$_{01}$ (saturation current, diode 1)', 
        'unit': 'A',
        'log': True
    },
    'I_02':  {
        'label': 'I$_{02}$ (saturation current, diode 2)', 
        'unit': 'A',     
        'log': True
    },
    'n':     {
        'label': 'n (ideality factor)',                    
        'unit': '',      
        'log': False
    },
    'N_A':   {
        'label': 'N$_A$ (acceptor density)',            
        'unit': 'cm$^{-3}$', 
        'log': False
    },
    'V_b':   {
        'label': 'V$_b$ (breakdown voltage)',
        'unit': 'V',    
        'log': False
    },
    'L_n':   {
        'label': 'L$_n$ (electron diffusion length)',
        'unit': 'um',    
        'log': False
    },
    'L_p':   {
        'label': 'L$_p$ (hole diffusion length)',
        'unit': 'um',    
        'log': False
    },
}

X_INFO = {
    'fluence': {
        'label': 'Fluence', 
        'unit': 'cm$^{-2}$', 
        'log': True
    },
    'energy (MeV)': {
        'label': 'Energy', 
        'unit': 'MeV', 
        'log': False
    },
}

## Interactive explorer

- **Particle type**: choose electron, proton, or both (overlaid)
- **X-axis**: fluence or energy
- **Y-axis parameter**: any of the 7 fitted parameters
- **Log scale** toggles are pre-set sensibly but can be overridden
- **Show table**: prints the filtered rows below the plot

In [ ]:
particle_options = ['both'] + sorted(df['particle type'].unique().tolist())

particle_w = widgets.ToggleButtons(
    options=particle_options,
    value='both',
    description='Particle:',
)

x_w = widgets.Dropdown(
    options=list(X_INFO.keys()),
    value='fluence',
    description='X-axis:',
)

y_w = widgets.Dropdown(
    options=list(PARAM_INFO.keys()),
    value='I_02',
    description='Y-axis:',
)

logx_w = widgets.Checkbox(
    value=True, 
    description='log X'
)
logy_w = widgets.Checkbox(
    value=True, 
    description='log Y'
)
table_w = widgets.Checkbox(
    value=False, 
    description='Show data table'
)

# energy filter only matters when x-axis = fluence (otherwise energy is the x-axis)
energy_w = widgets.SelectMultiple(
    options=sorted(df['energy (MeV)'].unique().tolist()),
    value=tuple(sorted(df['energy (MeV)'].unique().tolist())),
    description='Energy (MeV)\nfilter:',
    layout=Layout(width='160px', height='60px'),
)

def update_log_defaults(*args):
    """When the user changes the y parameter, suggest a sensible log scale,
    but don't fight them if they've already touched the checkbox manually."""
    logy_w.value = PARAM_INFO[y_w.value]['log']

y_w.observe(update_log_defaults, names='value')

def plot_explorer(
    particle, 
    x_col, 
    y_col, 
    logx, 
    logy, 
    show_table, 
    energies
):
    sub = df.copy()
    if particle != 'both':
        sub = sub[sub['particle type'] == particle]
    if x_col == 'fluence':
        sub = sub[sub['energy (MeV)'].isin(energies)]

    sub = sub.dropna(subset=[y_col])

    if sub.empty:
        print("No data for this combination.")
        return

    fig, ax = plt.subplots(figsize=(7, 5))

    for ptype, grp in sub.groupby('particle type'):
        if x_col == 'fluence':
            for energy, grp2 in grp.groupby('energy (MeV)'):
                grp2 = grp2.sort_values('fluence')
                ax.plot(
                    grp2[x_col], grp2[y_col],
                    marker=MeV_MARKERS.get(energy, 'o'),
                    color=COLORS.get(ptype, 'gray'),
                    linestyle='-',
                    label=f"{ptype}, {energy} MeV",
                )
        else:
            grp = grp.sort_values(x_col)
            ax.plot(
                grp[x_col], grp[y_col],
                marker='o',
                color=COLORS.get(ptype, 'gray'),
                linestyle='-',
                label=ptype,
            )

    xinfo = X_INFO[x_col]
    yinfo = PARAM_INFO[y_col]
    xlabel = f"{xinfo['label']}" + (f" ({xinfo['unit']})" if xinfo['unit'] else "")
    ylabel = f"{yinfo['label']}" + (f" ({yinfo['unit']})" if yinfo['unit'] else "")

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{yinfo['label']} vs {xinfo['label']}")

    if logx:
        #  guard against log(0)
        if (sub[x_col] <= 0).any():
            ax.set_xscale('symlog')
        else:
            ax.set_xscale('log')

    if logy:
        if (sub[y_col] <= 0).any():
            ax.set_yscale('symlog')
        else:
            ax.set_yscale('log')

    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8, loc='best')
    plt.tight_layout()
    plt.show()

    if show_table:
        cols = ['particle type', 'energy (MeV)', 'fluence', y_col]
        display(sub[cols].sort_values(['particle type', 'energy (MeV)', 'fluence']).reset_index(drop=True))

ui = widgets.VBox([
    widgets.HBox([particle_w]),
    widgets.HBox([x_w, y_w]),
    widgets.HBox([logx_w, logy_w, table_w]),
    widgets.HBox([energy_w]),
])

out = widgets.interactive_output(
    plot_explorer,
    {
        'particle': particle_w,
        'x_col': x_w,
        'y_col': y_w,
        'logx': logx_w,
        'logy': logy_w,
        'show_table': table_w,
        'energies': energy_w,
    },
)

display(ui, out)

## Side by side comparison

Pick two y-parameters to view as small multiples for quick comparison
across all particle/energy combinations, both vs. fluence.

In [ ]:
y1_w = widgets.Dropdown(
    options=list(PARAM_INFO.keys()), 
    value='n', 
    description='Param 1:'
)
y2_w = widgets.Dropdown(
    options=list(PARAM_INFO.keys()), 
    value='L_n', 
    description='Param 2:'
)


def plot_compare(y1, y2):
    _, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, y_col in zip(axes, [y1, y2]):
        sub = df.dropna(subset=[y_col])
        for (ptype, energy), grp in sub.groupby(['particle type', 'energy (MeV)']):
            grp = grp.sort_values('fluence')
            ax.plot(
                grp['fluence'], grp[y_col],
                marker=MeV_MARKERS.get(energy, 'o'),
                color=COLORS.get(ptype, 'gray'),
                label=f"{ptype}, {energy} MeV",
            )
        yinfo = PARAM_INFO[y_col]
        ax.set_xlabel('Fluence (cm$^{-2}$)')
        ax.set_ylabel(f"{yinfo['label']}" + (f" ({yinfo['unit']})" if yinfo['unit'] else ""))
        ax.set_title(yinfo['label'])

        if (sub['fluence'] <= 0).any():
            ax.set_xscale('symlog')
        else:
            ax.set_xscale('log')

        if yinfo['log'] and not (sub[y_col] <= 0).any():
            ax.set_yscale('log')

        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7)

    plt.tight_layout()
    plt.show()


compare_out = widgets.interactive_output(
    plot_compare,
    {
        'y1': y1_w, 
        'y2': y2_w
    }
)
display(widgets.HBox([y1_w, y2_w]), compare_out)